In [ ]:
%pip install seaborn

Function definitions 

In [1]:
# extract_xai_metrics.py
import argparse, os, glob, json, torch, pandas as pd
import torch.nn.functional as F
from torch_geometric.loader import DataLoader
from torch.utils.data import Subset
import networkx as nx
from sklearn.metrics import f1_score, roc_auc_score
from scipy.stats import spearmanr
import numpy as np

from util  import LinkPredictionDataset
from model import LinkPredictionAgentNet

def load_shared_resources(dataset_name):
    dataset_path = '{dataset_name}.pkl'.format(dataset_name=dataset_name)
    splits_path = '{dataset_name}_splits.pkl'.format(dataset_name=dataset_name)
    with open(dataset_path, 'rb') as f:
        dataset = pickle.load(f)
    with open(splits_path, 'rb') as f:
        splits = pickle.load(f)
    return dataset, splits

def freq_from_paths(paths_dict, data):
    """
    Args:
        paths_dict : {step_idx: LongTensor[num_agents]} (step 0 is src/tgt)
        num_agents : int
        num_nodes  : int, total #nodes in the subgraph (from data.x)

    Returns:
        freq : torch.FloatTensor[num_nodes], normalized frequency vector
    """
    step_tensors = [pos for step, pos in paths_dict.items() if step > 0]
    positions    = torch.cat(step_tensors)   # [A*T]
    T = len(step_tensors)

    num_agents = len(paths_dict[0])
    num_nodes = int(data.edge_index.max()) + 1 

    # Count visits for each node in subgraph
    visits = torch.bincount(positions.cpu(), minlength=num_nodes)

    freq = visits.float() / (num_agents * T)
    return freq

def kl_divergence(p, q, eps=1e-12):
    """KL(p||q) where p, q are 1-D numpy arrays summing to 1"""
    p = np.clip(p, eps, 1)
    q = np.clip(q, eps, 1)
    return float((p * np.log(p / q)).sum())
    
# ----------------------------------------------------------------------
def energy_from_paths_dict(paths_dict, num_agents):
    """
    Args
    ----
    paths_dict : {step_idx : LongTensor[num_agents]}
                 Keys start at 0.  Step-0 (source/target) is ignored.
    num_agents : int

    Returns
    -------
    energy : {node_id: E(n)}
             with E(n) = (visits / (A * T_eff))²
    """
    #collect position for steps 1 to T+1 (the first one is ignored since they are the source and target nodes)
    step_tensors = [pos for step, pos in paths_dict.items() if step > 0]

    positions = torch.cat(step_tensors)           # shape [T * A]
    T = len(step_tensors)                 # we skipped step 0

    # count number of visits per node
    num_nodes  = int(positions.max())
    visits  = torch.bincount(positions.cpu(), minlength=num_nodes+1)

    #normalise and square
    freq    = visits.float() / (T * num_agents)
    energy  = freq.pow(2)

    # convert to dict (and ignore the 0 energy ones)
    return {node: float(e) for node, e in enumerate(energy) if e > 0}


def degrees_within_subgraph(data):
    """
    data: a single torch_geometric Data object
    returns dict {node_id : (in_deg, out_deg, total_deg)}
    """
    edge_index = data.edge_index
    num_nodes = int(edge_index.max()) + 1 
    
    in_deg  = torch.bincount(edge_index[1], minlength=num_nodes)
    out_deg = torch.bincount(edge_index[0], minlength=num_nodes)
    tot = in_deg + out_deg
    return {int(i): (int(i_d), int(o_d), int(t_d))
            for i, (i_d, o_d, t_d) in enumerate(zip(in_deg, out_deg, tot))}
# ----------------------------------------------------------------------
def best_epoch_for_each_split(metrics_csv):
    """
    Returns a dict {split_id: best_epoch} where best_epoch is the
    epoch that maximises test_f1 WITHIN that split.
    """
    df = pd.read_csv(metrics_csv)

    # idxmax on each split, then pull the epoch column
    best_epochs = (
        df.loc[df.groupby("split")["test_f1"].idxmax()]
          .set_index("split")["epoch"]
          .to_dict()
    )
    print("Per-split best epochs:", best_epochs)   # e.g. {0: 92, 1: 87, 2: 101}
    return best_epochs

def load_checkpoint(model, ckpt_dir, epoch, device):
    ckpt = torch.load(os.path.join(ckpt_dir, f"epoch_{epoch}.pt"), map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"], strict=True)

# ----------------------------------------------------------------------
def main(args, dataset, splits):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    readout_type = args.readout_type
    if readout_type == 'final_only':
        args.final_readout_only = True
        args.use_step_readout_lin = False
        args.readout_mlp = False
    elif readout_type == 'all_steps':
        args.final_readout_only = False
        args.use_step_readout_lin = False
        args.readout_mlp = False
    elif readout_type == 'all_steps_linear':
        args.final_readout_only = False
        args.use_step_readout_lin = True
        args.readout_mlp = False
    elif readout_type == 'all_steps_mlp':
        args.final_readout_only = False
        args.use_step_readout_lin = False
        args.readout_mlp = True

    # reading best epoch
    best_epoch_dict = best_epoch_for_each_split(f'{args.output_dir}/metrics_per_split_epoch.csv')

    rows = []   # we’ll dump into a DataFrame
    # ------------------------------------------------------------------
    for split_id, (_, test_idx) in enumerate(splits):
        best_epoch = best_epoch_dict[split_id] 
        print(f"\n=== Split {split_id}  (best epoch {best_epoch}) ===")
        # Re-instantiate model with *exact* args used in training
        
        model = LinkPredictionAgentNet(num_features=dataset.num_features, hidden_units=args.hidden_units, num_out_classes=1, dropout=args.dropout, num_steps=args.num_steps,
                    num_agents=args.num_agents, reduce=args.reduce, node_readout=args.node_readout, use_step_readout_lin=args.use_step_readout_lin,
                    num_pos_attention_heads=args.num_pos_attention_heads, readout_mlp=args.readout_mlp, self_loops=args.self_loops, post_ln=args.post_ln,
                    attn_dropout=args.attn_dropout, no_time_cond=args.no_time_cond, mlp_width_mult=args.mlp_width_mult, activation_function=args.activation_function,
                    negative_slope=args.negative_slope, input_mlp=args.input_mlp, attn_width_mult=args.attn_width_mult,
                    random_agent=args.random_agent, test_argmax=args.test_argmax, global_agent_pool=args.global_agent_pool, agent_global_extra=args.agent_global_extra,
                    basic_global_agent=args.basic_global_agent, basic_agent=args.basic_agent, bias_attention=args.bias_attention, visited_decay=args.visited_decay,
                    sparse_conv=args.sparse_conv, mean_pool_only=args.mean_pool_only, edge_negative_slope=args.edge_negative_slope,
                    final_readout_only=args.final_readout_only, num_edge_features=args.num_edge_features)

        model = model.to(device)

        load_checkpoint(model, os.path.join(args.checkpoint_dir, f"split_{split_id}"), best_epoch, device)
        
                # ---------- build BOTH policies ----------
        # learned policy (original)
        model_learn = LinkPredictionAgentNet(num_features=dataset.num_features, hidden_units=args.hidden_units, num_out_classes=1, dropout=args.dropout, num_steps=args.num_steps,
                    num_agents=args.num_agents, reduce=args.reduce, node_readout=args.node_readout, use_step_readout_lin=args.use_step_readout_lin,
                    num_pos_attention_heads=args.num_pos_attention_heads, readout_mlp=args.readout_mlp, self_loops=args.self_loops, post_ln=args.post_ln,
                    attn_dropout=args.attn_dropout, no_time_cond=args.no_time_cond, mlp_width_mult=args.mlp_width_mult, activation_function=args.activation_function,
                    negative_slope=args.negative_slope, input_mlp=args.input_mlp, attn_width_mult=args.attn_width_mult,
                    random_agent=False, test_argmax=args.test_argmax, global_agent_pool=args.global_agent_pool, agent_global_extra=args.agent_global_extra,
                    basic_global_agent=args.basic_global_agent, basic_agent=args.basic_agent, bias_attention=args.bias_attention, visited_decay=args.visited_decay,
                    sparse_conv=args.sparse_conv, mean_pool_only=args.mean_pool_only, edge_negative_slope=args.edge_negative_slope,
                    final_readout_only=args.final_readout_only, num_edge_features=args.num_edge_features)
        
        model_learn.load_state_dict(model.state_dict())       # same weights
        model_learn = model_learn.to(device).eval()

        # random-walk policy
        model_rw = LinkPredictionAgentNet(num_features=dataset.num_features, hidden_units=args.hidden_units, num_out_classes=1, dropout=args.dropout, num_steps=args.num_steps,
                    num_agents=args.num_agents, reduce=args.reduce, node_readout=args.node_readout, use_step_readout_lin=args.use_step_readout_lin,
                    num_pos_attention_heads=args.num_pos_attention_heads, readout_mlp=args.readout_mlp, self_loops=args.self_loops, post_ln=args.post_ln,
                    attn_dropout=args.attn_dropout, no_time_cond=args.no_time_cond, mlp_width_mult=args.mlp_width_mult, activation_function=args.activation_function,
                    negative_slope=args.negative_slope, input_mlp=args.input_mlp, attn_width_mult=args.attn_width_mult,
                    random_agent=True, test_argmax=args.test_argmax, global_agent_pool=args.global_agent_pool, agent_global_extra=args.agent_global_extra,
                    basic_global_agent=args.basic_global_agent, basic_agent=args.basic_agent, bias_attention=args.bias_attention, visited_decay=args.visited_decay,
                    sparse_conv=args.sparse_conv, mean_pool_only=args.mean_pool_only, edge_negative_slope=args.edge_negative_slope,
                    final_readout_only=args.final_readout_only, num_edge_features=args.num_edge_features)
        model_rw.load_state_dict(model.state_dict())          # identical weights
        model_rw = model_rw.to(device).eval()

        test_ds = Subset(dataset, test_idx.tolist())
        loader  = DataLoader(test_ds, batch_size=1, shuffle=False)

        probs_all_learn = []
        probs_all_rw = []
        preds_all_learn = []
        preds_all_rw = []
        labels_all = []
        n = 0
        correct_learn = 0
        correct_rw = 0
        
        # ---------- iterate test samples ----------
        for local_i, data in enumerate(loader):
            data = data.to(device)
            node_pair = data.node_pair
            edge_attr = getattr(data, "edge_attr", None)

            with torch.no_grad():
                # learned policy
                logit_learn, paths_learn = model_learn(
                    x=data.x, edge_index=data.edge_index, batch=data.batch,
                    node_pair=node_pair, edge_feat=edge_attr, return_paths=True)

                # random walk policy
                logit_rw, paths_rw = model_rw(
                    x=data.x, edge_index=data.edge_index, batch=data.batch,
                    node_pair=node_pair, edge_feat=edge_attr, return_paths=True)

            # -------------- per-node freq / energy --------------
            p_learn = freq_from_paths(paths_learn, data)       # tensor
            p_rw    = freq_from_paths(paths_rw, data)

            S_learn = set((p_learn > 0).nonzero(as_tuple=True)[0].tolist())
            S_rw    = set((p_rw    > 0).nonzero(as_tuple=True)[0].tolist())

            energy_learn = p_learn.pow(2) # calculating node visitation energy
            energy_rw = p_rw.pow(2)

            numer = torch.minimum(p_learn, p_rw).sum()
            denom = torch.maximum(p_learn, p_rw).sum()
            jaccard_w = (numer / denom).item()          # also 0‒1

            # ---------- per-sample metrics ----------

            labels = data.y.float()  # BCE expects float labels
            n += labels.numel()

            probs_learn = torch.sigmoid(logit_learn)
            preds_learn = (probs_learn > args.threshold).long()
            correct_learn += preds_learn.eq(labels.long()).sum().item()
            
            probs_all_learn.extend(probs_learn.unsqueeze(0))      # this gives a 1D tensor of shape [1]
            preds_all_learn.extend(preds_learn.unsqueeze(0))
            labels_all.append(labels.unsqueeze(0))

            probs_rw = torch.sigmoid(logit_rw)
            preds_rw = (probs_rw > args.threshold).long()
            correct_rw += preds_rw.eq(labels.long()).sum().item()

            probs_all_rw.extend(probs_rw.unsqueeze(0))      # this gives a 1D tensor of shape [1]
            preds_all_rw.extend(preds_rw.unsqueeze(0))

            # node-wise degree dict
            degree_dict = degrees_within_subgraph(data)

            # ablation importance (logit drop) for learned walk
            delta_learn = []
            for k in range(len(p_learn)):
                if p_learn[k] == 0:           # skip unused nodes
                    delta_learn.append(0.0)
                    continue
                x_mask = data.x.clone()
                x_mask[k] = 0
                l_masked, _ = model_learn(x=x_mask, edge_index=data.edge_index,
                                          batch=data.batch, node_pair=node_pair,
                                          edge_feat=edge_attr, return_paths=False)
                delta_learn.append(float(torch.abs(logit_learn - l_masked)))

            # ablation importance for random walk
            delta_rw = []
            for k in range(len(p_learn)):
                if p_rw[k] == 0:           # skip unused nodes
                    delta_rw.append(0.0)
                    continue
                x_mask = data.x.clone()
                x_mask[k] = 0
                l_masked, _ = model_rw(x=x_mask, edge_index=data.edge_index,
                                          batch=data.batch, node_pair=node_pair,
                                          edge_feat=edge_attr, return_paths=False)
                delta_rw.append(float(torch.abs(logit_rw - l_masked)))
            
            # --- Learned policy correlation ---
            mask_vis_learn = energy_learn > 0
            energy_vals_learn = energy_learn[mask_vis_learn].cpu().numpy()
            delta_vals_learn = np.array(delta_learn)[mask_vis_learn]
            
            if len(np.unique(energy_vals_learn)) < 2 or len(np.unique(delta_vals_learn)) < 2:
                rho_energy_delta_learn = np.nan
            else:
                rho_energy_delta_learn = spearmanr(energy_vals_learn, delta_vals_learn).correlation
            
            # --- Random-walk policy correlation ---
            mask_vis_rw = energy_rw > 0
            energy_vals_rw = energy_rw[mask_vis_rw].cpu().numpy()
            delta_vals_rw = np.array(delta_rw)[mask_vis_rw]
            
            if len(np.unique(energy_vals_rw)) < 2 or len(np.unique(delta_vals_rw)) < 2:
                rho_energy_delta_rw = np.nan
            else:
                rho_energy_delta_rw = spearmanr(energy_vals_rw, delta_vals_rw).correlation

            # -------------- write rows --------------
            for n_id in range(len(p_learn)):
                in_d, out_d, tot_d = degree_dict.get(n_id, (0, 0, 0))
                rows.append({
                    "split"        : split_id,
                    "sample_idx"   : int(test_idx[local_i]),
                    "node_id"      : n_id,
                    "freq_learn"   : float(p_learn[n_id]),
                    "freq_rw"      : float(p_rw[n_id]),
                    "in_degree"    : in_d,
                    "out_degree"   : out_d,
                    "tot_degree"   : tot_d,
                    "energy_learn" : float(energy_learn[n_id]),
                    "energy_rw" : float(energy_rw[n_id]),
                    "jaccard_w"    : jaccard_w,
                    "rho_energy_delta_learn": rho_energy_delta_learn,
                    "rho_energy_delta_rw": rho_energy_delta_rw,
                    "global_f1_learn" : None,
                    "global_auc_learn": None,
                    "global_acc_learn": None,
                    "global_f1_rw"    : None,
                    "global_auc_rw"   : None,
                    "global_acc_rw"   : None,
                })
        
        probs_all_learn = torch.cat(probs_all_learn, dim=0)
        preds_all_learn = torch.cat(preds_all_learn, dim=0)

        probs_all_rw = torch.cat(probs_all_rw, dim=0)
        preds_all_rw = torch.cat(preds_all_rw, dim=0)

        labels_all = torch.cat(labels_all, dim=0)
            
        auc_learn = roc_auc_score(labels_all.cpu().numpy(), probs_all_learn.cpu().numpy())
        f1_learn = f1_score(labels_all.cpu().numpy(), preds_all_learn.cpu().numpy())
        acc_learn = correct_learn / n

        auc_rw = roc_auc_score(labels_all.cpu().numpy(), probs_all_rw.cpu().numpy())
        f1_rw = f1_score(labels_all.cpu().numpy(), preds_all_rw.cpu().numpy())
        acc_rw = correct_rw / n

        for row in rows:
            if row["split"] == split_id:
                row["global_f1_learn"] = f1_learn
                row["global_auc_learn"] = auc_learn
                row["global_acc_learn"] = acc_learn
                row["global_f1_rw"] = f1_rw
                row["global_auc_rw"] = auc_rw
                row["global_acc_rw"] = acc_rw
        
    # ------------------------------------------------------------------
    df = pd.DataFrame(rows)
    os.makedirs(args.output_dir, exist_ok=True)
    out_file = os.path.join(args.output_dir, "learn_vs_rw.csv")
    df.to_csv(out_file, index=False)
    print(f"\nSaved {len(df)} rows to {out_file}")

Loading arguments for GEPhilNet

In [4]:
import pickle
from optuna import Trial
from model import add_model_args
from util import set_seed
import optuna

# Load the study
storage = "sqlite:///optuna_shared_GePhil_2.db"
study_name = "AgentNet-Tuning-GePhil_2"
study = optuna.load_study(study_name=study_name, storage=storage)

# Get best trial and extract parameters
best_trial = study.best_trial
best_params = best_trial.params

set_seed(42)

parser = add_model_args(None, hyper=False)

# Suggest hyperparameters
args = parser.parse_args([])  # Empty because we'll override manually
args.threshold = best_params['threshold']
args.num_agents = best_params['num_agents']
args.batch_size = best_params['batch_size']
args.hidden_units = best_params['hidden_units']
args.num_steps = best_params['num_steps']
args.lr = best_params['lr']
args.dropout = best_params['dropout']

args.num_pos_attention_heads = best_params['num_pos_attention_heads']
args.gumbel_temp = best_params['gumbel_temp']

args.readout_type = best_params['readout_type']

# Fixed settings
args.epochs = 200
args.dataset = 'GePhil'
args.reduce = 'sum'
args.global_agent_pool = True
args.agents_global_extra = True
args.basic_global_agent = True
args.bias_attention = True
args.self_loops = True
# args.verbose = True
args.num_edge_features = 2
args.checkpoint_dir = './GePhil_checkpoint'
args.output_dir = './GePhil_output'
# args.n_splits = 2 # we have already split and saved the dataset

dataset, splits = load_shared_resources(args.dataset)

main(args, dataset, splits)

Per-split best epochs: {0: 18, 1: 29}

=== Split 0  (best epoch 18) ===

=== Split 1  (best epoch 29) ===

Saved 36021 rows to ./GePhil_output/learn_vs_rw.csv


In [3]:
best_params

{'threshold': 0.25419299197487066,
 'num_agents': 6,
 'batch_size': 16,
 'hidden_units': 32,
 'num_steps': 3,
 'lr': 0.0005800590530585434,
 'dropout': 0.021033046615779767,
 'num_pos_attention_heads': 1,
 'gumbel_temp': 0.4702250314136329,
 'readout_type': 'all_steps_linear'}

In [94]:
# 1.1 Reconstruct per-sample stats from per-node rows

df = pd.read_csv('GePhil_output/learn_vs_rw.csv')

sample_summary = (
    df.groupby(["split", "sample_idx"])
      .agg(
          jaccard_w=("jaccard_w", "first"),
          rho_learn=("rho_energy_delta_learn", "first"),
          rho_rw=("rho_energy_delta_rw", "first"),
          acc_learn=("global_acc_learn", "first"),
          acc_rw=("global_acc_rw", "first"),
          auc_learn=("global_auc_learn", "first"),
          auc_rw=("global_auc_rw", "first"),
          f1_learn=("global_f1_learn", "first"),
          f1_rw=("global_f1_rw", "first")
      )
)

# 1.2 Compute performance deltas
sample_summary["dAcc"] = sample_summary.acc_learn - sample_summary.acc_rw
sample_summary["dAuc"] = sample_summary.auc_learn - sample_summary.auc_rw
sample_summary["dF1"]  = sample_summary.f1_learn  - sample_summary.f1_rw
sample_summary["dRho"] = sample_summary.rho_learn - sample_summary.rho_rw

# 1.3 Show stats
print("\n=== GEPhilNet Sample-level XAI summary ===")
print(sample_summary.describe())


=== GEPhilNet Sample-level XAI summary ===
        jaccard_w   rho_learn      rho_rw   acc_learn      acc_rw   auc_learn  \
count  404.000000  402.000000  404.000000  404.000000  404.000000  404.000000   
mean     0.224185    0.097821    0.070999    0.821782    0.811881    0.896716   
std      0.105886    0.345412    0.371260    0.004957    0.009913    0.009767   
min      0.000000   -0.810163   -0.840083    0.816832    0.801980    0.886961   
25%      0.161290   -0.132639   -0.187707    0.816832    0.801980    0.886961   
50%      0.200000    0.100315    0.076721    0.821782    0.811881    0.896716   
75%      0.285714    0.341926    0.355844    0.826733    0.821782    0.906471   
max      0.714286    0.941124    0.882735    0.826733    0.821782    0.906471   

           auc_rw    f1_learn       f1_rw        dAcc        dAuc         dF1  \
count  404.000000  404.000000  404.000000  404.000000  404.000000  404.000000   
mean     0.899265    0.836969    0.828079    0.009901   -0.00254

In [88]:
from scipy.stats import wilcoxon
print("ΔF1 p-value:", wilcoxon(sample_summary.f1_learn, sample_summary.f1_rw).pvalue)
print("ΔAcc p-value:", wilcoxon(sample_summary.acc_learn, sample_summary.acc_rw).pvalue)
print("ΔAuc p-value:", wilcoxon(sample_summary.auc_learn, sample_summary.auc_rw).pvalue)

ΔF1 p-value: 2.4795409564780374e-72
ΔAcc p-value: 2.935329888687574e-19
ΔAuc p-value: 2.935329888687574e-19


## KarateClub

In [90]:
import optuna
import pickle
from optuna import Trial
from model import add_model_args
from util import set_seed

dataset = 'KarateLink'

# Load the study
storage = f"sqlite:///optuna_shared_{dataset}.db"
study_name = f"AgentNet-Tuning-{dataset}"
study = optuna.load_study(study_name=study_name, storage=storage)

# Get best trial and extract parameters
best_trial = study.best_trial
best_params = best_trial.params

set_seed(42)

parser = add_model_args(None, hyper=False)

# Suggest hyperparameters
args = parser.parse_args([])  # Empty because we'll override manually
args.threshold = best_params['threshold']
args.num_agents = best_params['num_agents']
args.batch_size = best_params['batch_size']
args.hidden_units = best_params['hidden_units']
args.num_steps = best_params['num_steps']
args.lr = best_params['lr']
args.dropout = best_params['dropout']

args.num_pos_attention_heads = best_params['num_pos_attention_heads']
args.gumbel_temp = best_params['gumbel_temp']

args.readout_type = best_params['readout_type']

# Fixed settings
args.epochs = 200
args.dataset = dataset
args.reduce = 'sum'
args.global_agent_pool = True
args.agents_global_extra = True
args.basic_global_agent = True
args.bias_attention = True
args.self_loops = True
# args.verbose = True
args.num_edge_features = 0
args.checkpoint_dir = f'./{dataset}_checkpoint'
args.output_dir = f'./{dataset}_output'
# args.n_splits = 2 # we have already split and saved the dataset

dataset, splits = load_shared_resources(args.dataset)

main(args, dataset, splits)

Per-split best epochs: {0: 82, 1: 64}

=== Split 0  (best epoch 82) ===

=== Split 1  (best epoch 64) ===

Saved 5125 rows to ./KarateLink_output/learn_vs_rw.csv


In [95]:
# 1.1 Reconstruct per-sample stats from per-node rows

df = pd.read_csv(f'KarateLink_output/learn_vs_rw.csv')

sample_summary = (
    df.groupby(["split", "sample_idx"])
      .agg(
          jaccard_w=("jaccard_w", "first"),
          rho_learn=("rho_energy_delta_learn", "first"),
          rho_rw=("rho_energy_delta_rw", "first"),
          acc_learn=("global_acc_learn", "first"),
          acc_rw=("global_acc_rw", "first"),
          auc_learn=("global_auc_learn", "first"),
          auc_rw=("global_auc_rw", "first"),
          f1_learn=("global_f1_learn", "first"),
          f1_rw=("global_f1_rw", "first")
      )
)

# 1.2 Compute performance deltas
sample_summary["dAcc"] = sample_summary.acc_learn - sample_summary.acc_rw
sample_summary["dAuc"] = sample_summary.auc_learn - sample_summary.auc_rw
sample_summary["dF1"]  = sample_summary.f1_learn  - sample_summary.f1_rw
sample_summary["dRho"] = sample_summary.rho_learn - sample_summary.rho_rw

# 1.3 Show stats
print("\n=== KarateLink Sample-level XAI summary ===")
print(sample_summary.describe())


=== KarateLink Sample-level XAI summary ===
        jaccard_w   rho_learn      rho_rw   acc_learn      acc_rw   auc_learn  \
count  156.000000  156.000000  156.000000  156.000000  156.000000  156.000000   
mean     0.340563    0.138189    0.166651    0.711538    0.705128    0.836949   
std      0.092811    0.250285    0.240340    0.083602    0.064309    0.024404   
min      0.148936   -0.350480   -0.408257    0.628205    0.641026    0.812623   
25%      0.285714   -0.052456    0.009701    0.628205    0.641026    0.812623   
50%      0.333333    0.128795    0.152965    0.711538    0.705128    0.836949   
75%      0.402597    0.333685    0.348170    0.794872    0.769231    0.861275   
max      0.542857    0.694046    0.739638    0.794872    0.769231    0.861275   

           auc_rw    f1_learn       f1_rw        dAcc        dAuc         dF1  \
count  156.000000  156.000000  156.000000  156.000000  156.000000  156.000000   
mean     0.840237    0.761905    0.741579    0.006410   -0.0032

In [93]:
from scipy.stats import wilcoxon
print("ΔF1 p-value:", wilcoxon(sample_summary.f1_learn, sample_summary.f1_rw).pvalue)
print("ΔAcc p-value:", wilcoxon(sample_summary.acc_learn, sample_summary.acc_rw).pvalue)
print("ΔAuc p-value:", wilcoxon(sample_summary.auc_learn, sample_summary.auc_rw).pvalue)

ΔF1 p-value: 4.796238000706577e-29
ΔAcc p-value: 2.741687967443589e-08
ΔAuc p-value: 2.741687967443589e-08
